# Zenodo Disaster Data Preprocessing

## Importing Packages

In [19]:
import os
import pandas as pd

## Reading Data

In [20]:
labels = {
    "no_disaster": 0,
    "earthquake": 1,
    "flood": 2,
    "hurricane": 3,
    "tornado": 4,
    "wildfire": 5
}

dfs = []

for file_name in [f for f in os.listdir("data/") if f.endswith(".ndjson")]:
    file_path = os.path.join("data/", file_name)
    label = labels[file_name.split("-")[0]]
    df = pd.read_json(file_path, lines=True)
    df["relevance"] = df["relevance"].replace(1, label)
    dfs.append(df)

In [21]:
combined_df = pd.concat(dfs, ignore_index=True)

In [22]:
combined_df.drop(columns=["id"], inplace=True)

In [23]:
combined_df.rename(columns={"relevance": "label"}, inplace=True)

In [24]:
combined_df.head()

,text,label
0,"earthquake in iloilo, philippines! my head's a...",1
1,new: felt <HASHTAG> earthquake <NUMBER> - boho...,1
2,"<NUMBER> earthquake, <NUMBER> m s of nueva vid...",1
3,just in: magnitude <NUMBER> earthquake: <NUMBE...,1
4,<NUMBER> earthquake recorded in the philippine...,1


In [25]:
combined_df.tail()

,text,label
128679,will it be easy? nope. will it be worth it? ab...,0
128680,<USER> all is cool here thank god!,0
128681,<USER> she's not adding up. she said she was <...,0
128682,"just listening to dave days' covers/originals,...",0
128683,"""anyone you're hoping to work with?"" demi: ""my...",0


In [26]:
combined_df.describe()

,label
count,128684.000000
mean,1.228327
std,1.460113
min,0.000000
25%,0.000000
50%,0.500000
75%,3.000000
max,5.000000


In [27]:
combined_df.isnull().sum()

text     0
label    0
dtype: int64

In [28]:
combined_df.nunique()

text     125019
label         6
dtype: int64

In [29]:
combined_df.dtypes

text     object
label     int64
dtype: object

In [30]:
df = combined_df

## Filtering Disaster Tweets

In [31]:
df = df[df["label"] != 0]

In [32]:
df.describe()

,label
count,64342.000000
mean,2.456654
std,1.116367
min,1.000000
25%,1.000000
50%,3.000000
75%,3.000000
max,5.000000


In [33]:
df.nunique()

text     62784
label        5
dtype: int64

## Removing Invalid Whitespace

In [34]:
if df["text"].str.contains(r"\s\s").any():
    print("Error: Consecutive whitespace.")
if df["text"].str.count(r"\s").sum() != df["text"].str.count(" ").sum():
    print("Error: Non-space whitespace.")

## Writing Data

In [35]:
df.to_csv("data/preprocessed_data.csv", sep="\t", encoding="utf-16", index=False)